# Iris Flower Clustering Project
Apply K-Means (k=3), visualize clusters, compare with true species, and use PCA for dimensionality reduction.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
os.makedirs('outputs', exist_ok=True)

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target
target_names = iris.target_names
df = X.copy()
df['true_species'] = y.map(dict(enumerate(target_names)))
print('Dataset shape:', X.shape)
display(df.head())

## 1. Standardize the features and apply K-Means

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
df['predicted_cluster'] = clusters
print('Silhouette score:', round(silhouette_score(X_scaled, clusters), 4))
print('Adjusted Rand Index:', round(adjusted_rand_score(y, clusters), 4))
print(df['predicted_cluster'].value_counts().sort_index())

## 2. Visualize the predicted clusters

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='petal length (cm)', y='petal width (cm)', hue='predicted_cluster', palette='Set1', s=75)
plt.title('Iris Flowers: K-Means Clusters (k=3)')
plt.show()

## 3. Compare actual species with predicted clusters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x='petal length (cm)', y='petal width (cm)', hue='true_species', palette='Set2', s=65, ax=axes[0])
axes[0].set_title('Actual Iris Species')
sns.scatterplot(data=df, x='petal length (cm)', y='petal width (cm)', hue='predicted_cluster', palette='Set1', s=65, ax=axes[1])
axes[1].set_title('Predicted K-Means Clusters')
plt.tight_layout()
plt.show()
display(pd.crosstab(df['true_species'], df['predicted_cluster']))

## 4. Apply PCA to reduce four dimensions to two

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['predicted_cluster'] = clusters
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='predicted_cluster', palette='Set1', s=75)
plt.title('K-Means Clusters after PCA (2D)')
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.show()
print('Total variance retained:', round(pca.explained_variance_ratio_.sum()*100, 2), '%')

## Conclusion
K-Means groups the Iris observations into three clusters without using species labels. The scatter plots help compare the discovered groups with the known species. PCA projects the four standardized measurements into two components for visualization. Cluster numbers are arbitrary, so compare group structure rather than expecting cluster 0 to equal species 0.